In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from crewai import Agent, Task, Crew, Process
from crewai import LLM

llm = LLM(
model='openai/gpt-4.1-mini',
temperature=0.7,
max_tokens=1000,
frequency_penalty=0.1,
presence_penalty=0.1,
top_p=0.9
)

### 여행 기획자 에이전트 정의

In [3]:
# LLM의 지식 기반으로 답변을 해주는 것. 최신 정보가 적용 안되어 있을 수도 있음. 그래서 tool에 대한게 중요!
# tool에 현재 날짜를 나타내는 datetime 등을 이용한 방법 or 프롬프트에 현 시점을 언급 or 사용자가 직접 날짜를

travel_agent = Agent(
    role = "여행 기획자",
    goal = "사용자의 요청에 따라 {place} 여행 일정을 계획하고 제안합니다.",
    backstory="여행사에서 10년 경력의 전문 여행 플래너로, 다양한 국내 여행 코스를 알고 있습니다.",
    llm=llm,
    verbose=True
)

### 3일 여행 일정 작성 Task 정의

In [4]:
itinerary_task = Task(
    description=(
        "{place}에서 {days}일간 여행 일정을 계획해주세요. 1일 이상이면 1일차, 2일차, 3일차로 나누고, 각 일자마다 아침/점심/저녁에 할 활동을 상세히 제안하세요."
        "여행 일정에는 {place}의 주요 관광지와 현지 맛집 추천을 포함하고, 교통 수단 정보나 팁이 있으면 함께 제공하세요."
    ),
    agent=travel_agent,
    expected_output="{days}일을 Day1, Day2, Day3 으로 구분된 상세 일정 제안"
)



### Crew 생성 및 실행 (순차 실행 - Task가 하나뿐이므로 순차 처리)

In [5]:
crew_single = Crew(
    agents=[travel_agent],
    tasks=[itinerary_task],
    process= Process.sequential,
    verbose=True
)

In [6]:
place = '강릉'
days = 4

print(f"=== [단일 에이전트] {place} {days}일 일정 생성 시작 ===")
result_single = await crew_single.kickoff_async(inputs = {"place":place, "days":days})

print(f"=== [단일 에이전트] 생성된 {place} {days}일 일정 ===")
print(result_single)

=== [단일 에이전트] 강릉 4일 일정 생성 시작 ===


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 73a0509d-a02b-44cd-b6da-d50bc0a7c877                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 강릉에서 4일간 여행 일정을 계획해주세요. 1일 이상이면 1일차, 2일차, 3일차로 나누고, 각 일자마다          │
│  아침/점심/저녁에 할 활동을 상세히 제안하세요.여행 일정에는 강릉의 주요 관광지와 현지 맛집 추천을 포함하고,     │
│  교통 수단 정보나 팁이 있으면 함께 제공하세요.                                                                  │
│  ID: 56bbb89a-d79f-4f23-858d-02714a2de577                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 여행 기획자                                                                                             │
│                                                                                                                 │
│  Task: 강릉에서 4일간 여행 일정을 계획해주세요. 1일 이상이면 1일차, 2일차, 3일차로 나누고, 각 일자마다          │
│  아침/점심/저녁에 할 활동을 상세히 제안하세요.여행 일정에는 강릉의 주요 관광지와 현지 맛집 추천을 포함하고,     │
│  교통 수단 정보나 팁이 있으면 함께 제공하세요.                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 여행 기획자                                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  강릉 4일 여행 일정 제안드립니다. 주요 관광지 방문과 현지 맛집 체험을 고루 포함했고, 이동 편의를 위한 교통      │
│  팁도 함께 안내해 드립니다.                                                                                     │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Day 1: 강릉 도착 및 시내 탐방                                                                              │
│                                                                                                                 │
│  - **아침**                                                                                                     │
│    - 서울 또는 출발지에서 강릉행 KTX 탑승 (서울역 → 강릉역, 약 2시간 소요)                                      │
│    - 강릉역 도착 후 숙소 체크인 (짐 보관 가능 여부 확인)                                                        │
│  - **점심**                                                                                                     │
│    - 강릉 중앙시장 내 ‘초당할머니순두부’ 방문                                                                   │
│      - 순두부찌개와 두부 요리 전문, 강릉 대표 맛집                                                              │
│  - **오후**                                                                                                     │
│    - 안목해변 카페거리 산책 및 커피 한잔                                                                        │
│      - 해변 바로 앞에 위치한 다양한 로스터리 카페들에서 바다 풍경 감상                                          │
│    - 경포호 산책 및 경포대 방문                                                                                 │
│      - 경포대에서 경포호와 동해 바다 조망                                                                       │
│  - **저녁**                                                                                                     │
│    - ‘초당순두부마을’ 근처의 ‘오죽헌’ 방문 (저녁 늦게까지 가능 여부 확인 필요)                                  │
│    - 저녁 식사는 ‘강릉교동짬뽕’에서 매운 해물짬뽕 추천                                                          │
│  - **교통 팁**                                                                                                  │
│    - 강릉 시내는 택시나 버스 이용 가능, 안목해변과 경포대는 택시로 10분 내외 거리                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Day 2: 자연과 문화 체험                                                                                    │
│                                                                                                                 │
│  - **아침**                                                                                                     │
│    - 숙소 근처 카페에서 아침 식사 (예: ‘테라로사 커피공장’ – 강릉 대표 커피 명소)                               │
│  - **점심**                                                                                                     │
│    - 주문진항으로 이동 (택시 또는 버스 약 30분)                                                                 │
│    - 주문진 수산시장 내 ‘횟집’에서 신선한 회 및 해산물 점심 식사                               

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 강릉에서 4일간 여행 일정을 계획해주세요. 1일 이상이면 1일차, 2일차, 3일차로 나누고, 각 일자마다          │
│  아침/점심/저녁에 할 활동을 상세히 제안하세요.여행 일정에는 강릉의 주요 관광지와 현지 맛집 추천을 포함하고,     │
│  교통 수단 정보나 팁이 있으면 함께 제공하세요.                                                                  │
│  Agent: 여행 기획자                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 73a0509d-a02b-44cd-b6da-d50bc0a7c877                                                                       │
│  Final Output: 강릉 4일 여행 일정 제안드립니다. 주요 관광지 방문과 현지 맛집 체험을 고루 포함했고, 이동 편의를  │
│  위한 교통 팁도 함께 안내해 드립니다.                                                                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Day 1: 강릉 도착 및 시내 탐방                                                                              │
│                                                                                                                 │
│  - **아침**                                                                                                     │
│    - 서울 또는 출발지에서 강릉행 KTX 탑승 (서울역 → 강릉역, 약 2시간 소요)                                      │
│    - 강릉역 도착 후 숙소 체크인 (짐 보관 가능 여부 확인)                                                        │
│  - **점심**                                                                                                     │
│    - 강릉 중앙시장 내 ‘초당할머니순두부’ 방문                                                                   │
│      - 순두부찌개와 두부 요리 전문, 강릉 대표 맛집                                                              │
│  - **오후**                                                                                                     │
│    - 안목해변 카페거리 산책 및 커피 한잔                                                                        │
│      - 해변 바로 앞에 위치한 다양한 로스터리 카페들에서 바다 풍경 감상                                          │
│    - 경포호 산책 및 경포대 방문                                                                                 │
│      - 경포대에서 경포호와 동해 바다 조망                                                                       │
│  - **저녁**                                                                                                     │
│    - ‘초당순두부마을’ 근처의 ‘오죽헌’ 방문 (저녁 늦게까지 가능 여부 확인 필요)                                  │
│    - 저녁 식사는 ‘강릉교동짬뽕’에서 매운 해물짬뽕 추천                                                          │
│  - **교통 팁**                                                                                                  │
│    - 강릉 시내는 택시나 버스 이용 가능, 안목해변과 경포대는 택시로 10분 내외 거리                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Day 2: 자연과 문화 체험                                                                                    │
│                                                                                                                 │
│  - **아침**                                                                                                     │
│    - 숙소 근처 카페에서 아침 식사 (예: ‘테라로사 커피공장’ – 강릉 대표 커피 명소)                               │
│  - **점심**                                                                                                     │
│    - 주문진항으로 이동 (택시 또는 버스 약 30분)                                                                 │
│    - 주문진 수산시장 내 ‘횟집’에서 신선한 회 및 해산물 점심 식사                         

=== [단일 에이전트] 생성된 강릉 4일 일정 ===
강릉 4일 여행 일정 제안드립니다. 주요 관광지 방문과 현지 맛집 체험을 고루 포함했고, 이동 편의를 위한 교통 팁도 함께 안내해 드립니다.

---

### Day 1: 강릉 도착 및 시내 탐방

- **아침**
  - 서울 또는 출발지에서 강릉행 KTX 탑승 (서울역 → 강릉역, 약 2시간 소요)
  - 강릉역 도착 후 숙소 체크인 (짐 보관 가능 여부 확인)
- **점심**
  - 강릉 중앙시장 내 ‘초당할머니순두부’ 방문
    - 순두부찌개와 두부 요리 전문, 강릉 대표 맛집
- **오후**
  - 안목해변 카페거리 산책 및 커피 한잔
    - 해변 바로 앞에 위치한 다양한 로스터리 카페들에서 바다 풍경 감상
  - 경포호 산책 및 경포대 방문
    - 경포대에서 경포호와 동해 바다 조망
- **저녁**
  - ‘초당순두부마을’ 근처의 ‘오죽헌’ 방문 (저녁 늦게까지 가능 여부 확인 필요)
  - 저녁 식사는 ‘강릉교동짬뽕’에서 매운 해물짬뽕 추천
- **교통 팁**
  - 강릉 시내는 택시나 버스 이용 가능, 안목해변과 경포대는 택시로 10분 내외 거리

---

### Day 2: 자연과 문화 체험

- **아침**
  - 숙소 근처 카페에서 아침 식사 (예: ‘테라로사 커피공장’ – 강릉 대표 커피 명소)
- **점심**
  - 주문진항으로 이동 (택시 또는 버스 약 30분)
  - 주문진 수산시장 내 ‘횟집’에서 신선한 회 및 해산물 점심 식사
- **오후**
  - 주문진 해변 산책 및 해양레저 체험 가능 시 체험 (카약, 패들보드 등)
  - 강릉 바우길 일부 구간 트레킹 (자연과 함께 걷기 좋은 코스)
- **저녁**
  - 강릉 시내 복합문화공간 ‘강릉문화예술센터’ 주변 맛집 탐방
  - ‘이성당 빵집’ 방문 후 간식 또는 저녁 대용으로 빵 구입 가능 (강릉 대표 빵집)
- **교통 팁**
  - 주문진행은 택시 이용이 편리하며, 버스 시간표 미리 확인 권장

---

### D

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯